Step 1 – Install and Import Libraries

In [2]:
!pip install pyspark

In [3]:
import pandas as pd
import random
import time
from pyspark.sql import SparkSession


Step 2 – Create Spark Session

In [4]:
spark = SparkSession.builder.appName("RealTimeLogProcessing").getOrCreate()

Step 3 – Create Log Data Generator

In [5]:
pages = ["Home","Products","Cart","Checkout","Login","Contact"]

status_codes = [200,200,200,404,500]

def generate_log():

    page = random.choice(pages)
    status = random.choice(status_codes)

    return {
        "Page": page,
        "Status": status
    }


Step 4 – Simulate Streaming Log Data

In [7]:
logs = []

for i in range(25):

    log = generate_log()

    logs.append(log)

    print("New Log:",log)

    time.sleep(1)


New Log: {'Page': 'Contact', 'Status': 200}
New Log: {'Page': 'Checkout', 'Status': 500}
New Log: {'Page': 'Login', 'Status': 200}
New Log: {'Page': 'Products', 'Status': 500}
New Log: {'Page': 'Contact', 'Status': 200}
New Log: {'Page': 'Login', 'Status': 500}
New Log: {'Page': 'Products', 'Status': 200}
New Log: {'Page': 'Products', 'Status': 500}
New Log: {'Page': 'Contact', 'Status': 200}
New Log: {'Page': 'Home', 'Status': 500}
New Log: {'Page': 'Contact', 'Status': 404}
New Log: {'Page': 'Products', 'Status': 200}
New Log: {'Page': 'Contact', 'Status': 200}
New Log: {'Page': 'Products', 'Status': 200}
New Log: {'Page': 'Home', 'Status': 200}
New Log: {'Page': 'Contact', 'Status': 200}
New Log: {'Page': 'Home', 'Status': 500}
New Log: {'Page': 'Checkout', 'Status': 200}
New Log: {'Page': 'Login', 'Status': 500}
New Log: {'Page': 'Contact', 'Status': 500}
New Log: {'Page': 'Home', 'Status': 500}
New Log: {'Page': 'Cart', 'Status': 500}
New Log: {'Page': 'Login', 'Status': 200}
New 

Step 5 – Convert Logs to DataFrame

In [8]:
log_df = pd.DataFrame(logs)

log_df


,Page,Status
0,Contact,200
1,Checkout,500
2,Login,200
3,Products,500
4,Contact,200
5,Login,500
6,Products,200
7,Products,500
8,Contact,200
9,Home,500


Step 6 – Convert Dataset to Spark DataFrame

In [9]:
spark_df = spark.createDataFrame(log_df)

spark_df.show()


+--------+------+
|    Page|Status|
+--------+------+
| Contact|   200|
|Checkout|   500|
|   Login|   200|
|Products|   500|
| Contact|   200|
|   Login|   500|
|Products|   200|
|Products|   500|
| Contact|   200|
|    Home|   500|
| Contact|   404|
|Products|   200|
| Contact|   200|
|Products|   200|
|    Home|   200|
| Contact|   200|
|    Home|   500|
|Checkout|   200|
|   Login|   500|
| Contact|   500|
+--------+------+
only showing top 20 rows


Step 7 – Real-Time Page Visit Analysis

In [10]:
page_visits = spark_df.groupBy("Page").count()

page_visits.show()


+--------+-----+
|    Page|count|
+--------+-----+
|    Home|    5|
|   Login|    5|
|Checkout|    2|
|Products|    5|
| Contact|    7|
|    Cart|    1|
+--------+-----+



Step 8 – Error Detection

In [11]:
errors = spark_df.filter(spark_df.Status != 200)

errors.show()


+--------+------+
|    Page|Status|
+--------+------+
|Checkout|   500|
|Products|   500|
|   Login|   500|
|Products|   500|
|    Home|   500|
| Contact|   404|
|    Home|   500|
|   Login|   500|
| Contact|   500|
|    Home|   500|
|    Cart|   500|
+--------+------+



Step 9 – Save Processed Logs

In [12]:
spark_df.toPandas().to_csv("processed_logs.csv",index=False)

print("Logs saved successfully")


Logs saved successfully
